# m06_l02_replay

Exobean Engineering. Run the cells from top to bottom.

In [ ]:
# Google Colab only: fetches the free code companion, then run the rest.
import io, os, sys, urllib.request, zipfile
URL = "https://github.com/exobean/ai-agents-level1/releases/latest/download/shopassist-companion.zip"
if "google.colab" in sys.modules and not os.path.isdir("shopassist"):
    try:
        data = io.BytesIO(urllib.request.urlopen(URL).read())
    except OSError:  # network blocks the download: upload the zip
        from google.colab import files
        data = next(iter(files.upload()))
    zipfile.ZipFile(data).extractall(".")
    os.chdir(next(d for d in os.listdir(".") if d.endswith("-companion")))
    sys.path.insert(0, os.getcwd())


## Replay

In [ ]:
import tempfile
from pathlib import Path

from shopassist.app_v2 import new_session
from shopassist.cassette import RecordingModel, ReplayMiss, ReplayModel
from shopassist.models import TeachingModel

tape = Path(tempfile.mkdtemp()) / "cassette.json"

# 1. Record once (in real life: the real, paid model).
rec, _ = new_session(RecordingModel(TeachingModel(), tape))
first = rec.ask("Where is my order #1042?")

# 2. Replay in every test run: no network, no cost, same answer.
replay, _ = new_session(ReplayModel(tape))
again = replay.ask("Where is my order #1042?")
print("replay matches:", again == first)

# 3. A changed conversation is caught, not silently answered.
try:
    new_session(ReplayModel(tape))[0].ask("Where is order #1043?")
except ReplayMiss as miss:
    print("replay miss:   ", miss)
